In [ ]:
from project2skeleton import nb_train, nb_test, f_score, load_data

# Load training data
x_train, y_train = load_data("./SPAM_training_set/")
# Train the model
model = nb_train(x_train, y_train)

# Load testing data
x_test, y_test = load_data("./SPAM_test_set/")

# Set up configurations for experiments
configurations = [
    {"use_log": False, "smoothing": False},
    {"use_log": False, "smoothing": True},
    {"use_log": True, "smoothing": False},
    {"use_log": True, "smoothing": True}
]

# Dictionary to hold results
results = {}
for config in configurations:
    # Perform testing with each configuration
    y_pred = nb_test(x_test, model, use_log=config["use_log"], smoothing=config["smoothing"])
    # Calculate F1 score
    f1 = f_score(y_test, y_pred)
    results[f"Log: {config['use_log']}, Smoothing: {config['smoothing']}"] = f1

# Print results
for config, f1_score in results.items():
    print(f"{config} -> F1-score: {f1_score:.4f}")



In [ ]:
import glob
import os
import math
import re


def load_data(directory):
    x = []
    y = []
    for f in glob.glob(os.path.join(directory, "HAM.*.txt")):
        with open(f, 'r') as file:
            x.append(file.read())
            y.append(0)
    for f in glob.glob(os.path.join(directory, "SPAM.*.txt")):
        with open(f, 'r') as file:
            x.append(file.read())
            y.append(1)
    return x, y

def nb_train(x, y):
    model = {
        'ham_count': 0,
        'spam_count': 0,
        'ham_fd': {},
        'spam_fd': {}
    }
    for document, label in zip(x, y):
        words = re.findall(r'\b\w+\b', document.lower())
        if label == 0:
            model['ham_count'] += 1
            update_frequency_distribution(words, model['ham_fd'])
        else:
            model['spam_count'] += 1
            update_frequency_distribution(words, model['spam_fd'])
    return model

def update_frequency_distribution(words, freq_dict):
    for word in words:
        if word in freq_dict:
            freq_dict[word] += 1
        else:
            freq_dict[word] = 1


def nb_test(docs, trained_model, use_log=False, smoothing=False):
    predictions = []
    # Calculate the prior probabilities
    total_emails = trained_model['ham_count'] + trained_model['spam_count']
    prior_ham = trained_model['ham_count'] / total_emails
    prior_spam = trained_model['spam_count'] / total_emails
    
    # Pre-calculate total words in ham and spam for smoothing
    total_ham_words = sum(trained_model['ham_fd'].values())
    total_spam_words = sum(trained_model['spam_fd'].values())
    vocab_size = len(set(trained_model['ham_fd']).union(set(trained_model['spam_fd'])))

    for doc in docs:
        words = re.findall(r'\b\w+\b', doc.lower())
        # Calculate the log probabilities or direct probabilities
        ham_prob = math.log(prior_ham) if use_log else prior_ham
        spam_prob = math.log(prior_spam) if use_log else prior_spam
        
        for word in words:
            if smoothing:
                # Add-one Laplace smoothing
                ham_word_prob = (trained_model['ham_fd'].get(word, 0) + 1) / (total_ham_words + vocab_size)
                spam_word_prob = (trained_model['spam_fd'].get(word, 0) + 1) / (total_spam_words + vocab_size)
            else:
                # No smoothing
                ham_word_prob = trained_model['ham_fd'].get(word, 0) / total_ham_words if trained_model['ham_fd'].get(word, 0) > 0 else 1e-10
                spam_word_prob = trained_model['spam_fd'].get(word, 0) / total_spam_words if trained_model['spam_fd'].get(word, 0) > 0 else 1e-10
            
            if use_log:
                ham_prob += math.log(ham_word_prob)
                spam_prob += math.log(spam_word_prob)
            else:
                ham_prob *= ham_word_prob
                spam_prob *= spam_word_prob
        
        # Compare the probabilities to classify
        if use_log:
            prediction = 0 if ham_prob > spam_prob else 1
        else:
            prediction = 0 if ham_prob >= spam_prob else 1
        
        predictions.append(prediction)
    
    return predictions
    
def f_score(y_true, y_pred):
    true_positives = sum(1 for true, pred in zip(y_true, y_pred) if true == pred == 1)
    false_positives = sum(1 for true, pred in zip(y_true, y_pred) if true == 0 and pred == 1)
    false_negatives = sum(1 for true, pred in zip(y_true, y_pred) if true == 1 and pred == 0)

    precision = true_positives / (true_positives + false_positives) if true_positives + false_positives > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if true_positives + false_negatives > 0 else 0
    
    if precision + recall == 0:
        return 0
    return 2 * (precision * recall) / (precision + recall)




if __name__ == "__main__":
    x_train, y_train = load_data("./SPAM_training_set/")
    model = nb_train(x_train, y_train)
    x_test, y_test = load_data("./SPAM_test_set/")
    configurations = [
        {"use_log": False, "smoothing": False},
        {"use_log": False, "smoothing": True},
        {"use_log": True, "smoothing": False},
        {"use_log": True, "smoothing": True}
    ]
    results = {}
    for config in configurations:
        y_pred = nb_test(x_test, model, use_log=config["use_log"], smoothing=config["smoothing"])
        f1 = f_score(y_test, y_pred)
        results[f"Log: {config['use_log']}, Smoothing: {config['smoothing']}"] = f1

    for config, f1_score in results.items():
        print(f"{config} -> F1-score: {f1_score:.4f}")



In [ ]:

import glob
import os
import numpy as np
from sklearn.metrics import f1_score

def load_data(directory):
  """
  Loads the email data from the specified directory.

  Args:
    directory: The directory containing the email files.

  Returns:
    A tuple containing two lists: x (documents) and y (labels).
  """
  x = []
  y = []
  for f in glob.glob(os.path.join(directory, "HAM.*.txt")):
    with open(f, 'r') as file:
      x.append(file.read())
      y.append(0)
  for f in glob.glob(os.path.join(directory, "SPAM.*.txt")):
    with open(f, 'r') as file:
      x.append(file.read())
      y.append(1)
  return x, y

def nb_train(x, y):
  """
  Trains a Naive Bayes classifier on the given data.

  Args:
    x: A list of documents (strings).
    y: A list of labels (0 for HAM, 1 for SPAM).

  Returns:
    A dictionary representing the trained model.
  """
  model = {
    'ham_count': 0,
    'spam_count': 0,
    'ham_fd': {},
    'spam_fd': {}
  }

  for label in y:
    if label == 0:
      model['ham_count'] += 1
    else:
      model['spam_count'] += 1

  for i, doc in enumerate(x):
    label = y[i]
    words = doc.split()
    for word in words:
      if label == 0:
        model['ham_fd'][word] = model['ham_fd'].get(word, 0) + 1
      else:
        model['spam_fd'][word] = model['spam_fd'].get(word, 0) + 1

  return model

def nb_test(docs, model, use_log=False, smoothing=False):
  """
  Tests a trained Naive Bayes classifier on new documents.

  Args:
    docs: A list of documents (strings).
    model: The trained Naive Bayes model.
    use_log: Whether to use log calculations (True) or multiplication (False).
    smoothing: Whether to use add-one smoothing (True) or not (False).

  Returns:
    A list of predicted labels (0 for HAM, 1 for SPAM).
  """
  predictions = []
  for doc in docs:
    words = doc.split()
    ham_prob = 0.0
    spam_prob = 0.0
    for word in words:
      ham_count = model['ham_fd'].get(word, 0)
      spam_count = model['spam_fd'].get(word, 0)
      if use_log:
        ham_prob += np.log((ham_count + smoothing) / (model['ham_count'] + len(model['ham_fd']) * smoothing))
        spam_prob += np.log((spam_count + smoothing) / (model['spam_count'] + len(model['spam_fd']) * smoothing))
      else:
        ham_prob *= (ham_count + smoothing) / (model['ham_count'] + len(model['ham_fd']) * smoothing)
        spam_prob *= (spam_count + smoothing) / (model['spam_count'] + len(model['spam_fd']) * smoothing)
    if ham_prob > spam_prob:
      predictions.append(0)
    else:
      predictions.append(1)
  return predictions

def f_score(y_true, y_pred):
  """
  Calculates the F1-score for the given predictions.

  Args:
    y_true: A list of true labels.
    y_pred: A list of predicted labels.

  Returns:
    The F1-score.
  """
  return f1_score(y_true, y_pred)

# Load the training and test data
x_train, y_train = load_data("./SPAM_training_set/")
x_test, y_test = load_data("./SPAM_test_set/")

# Train the model
model = nb_train(x_train, y_train)

# Run the experiments
print("F1-score without log and without smoothing:", f_score(y_test, nb_test(x_test, model, use_log=False, smoothing=False)))
print("F1-score with log and without smoothing:", f_score(y_test, nb_test(x_test, model, use_log=True, smoothing=False)))
print("F1-score without log and with smoothing:", f_score(y_test, nb_test(x_test, model, use_log=False, smoothing=True)))
print("F1-score with log and with smoothing:", f_score(y_test, nb_test(x_test, model, use_log=True, smoothing=True)))



In [ ]:
import glob
import os
import math
import re

def load_data(directory):
    x = []
    y = []
    for f in glob.glob(os.path.join(directory, "HAM.*.txt")):
        with open(f, 'r') as file:
            x.append(file.read())
            y.append(0)
    for f in glob.glob(os.path.join(directory, "SPAM.*.txt")):
        with open(f, 'r') as file:
            x.append(file.read())
            y.append(1)
    return x, y

def nb_train(x, y):
    model = {
        'ham_count': 0,
        'spam_count': 0,
        'ham_fd': {},
        'spam_fd': {}
    }
    for document, label in zip(x, y):
        words = re.findall(r'\b\w+\b', document.lower())
        if label == 0:
            model['ham_count'] += 1
            update_frequency_distribution(words, model['ham_fd'])
        else:
            model['spam_count'] += 1
            update_frequency_distribution(words, model['spam_fd'])
    return model

def update_frequency_distribution(words, freq_dict):
    for word in words:
        if word in freq_dict:
            freq_dict[word] += 1
        else:
            freq_dict[word] = 1


def nb_test(docs, trained_model, use_log=False, smoothing=False):
    predictions = []
    # Calculate the prior probabilities
    total_emails = trained_model['ham_count'] + trained_model['spam_count']
    prior_ham = trained_model['ham_count'] / total_emails
    prior_spam = trained_model['spam_count'] / total_emails
    
    # Pre-calculate total words in ham and spam for smoothing
    total_ham_words = sum(trained_model['ham_fd'].values())
    total_spam_words = sum(trained_model['spam_fd'].values())
    vocab_size = len(set(trained_model['ham_fd']).union(set(trained_model['spam_fd'])))

    for doc in docs:
        words = re.findall(r'\b\w+\b', doc.lower())
        # Calculate the log probabilities or direct probabilities
        ham_prob = math.log(prior_ham) if use_log else prior_ham
        spam_prob = math.log(prior_spam) if use_log else prior_spam
        
        for word in words:
            if smoothing:
                # Add-one Laplace smoothing
                ham_word_prob = (trained_model['ham_fd'].get(word, 0) + 1) / (total_ham_words + vocab_size)
                spam_word_prob = (trained_model['spam_fd'].get(word, 0) + 1) / (total_spam_words + vocab_size)
            else:
                # No smoothing
                ham_word_prob = trained_model['ham_fd'].get(word, 0) / total_ham_words if trained_model['ham_fd'].get(word, 0) > 0 else 1e-10
                spam_word_prob = trained_model['spam_fd'].get(word, 0) / total_spam_words if trained_model['spam_fd'].get(word, 0) > 0 else 1e-10
            
            if use_log:
                ham_prob += math.log(ham_word_prob)
                spam_prob += math.log(spam_word_prob)
            else:
                ham_prob *= ham_word_prob
                spam_prob *= spam_word_prob
        
        # Compare the probabilities to classify
        if use_log:
            prediction = 0 if ham_prob > spam_prob else 1
        else:
            prediction = 0 if ham_prob >= spam_prob else 1
        
        predictions.append(prediction)
    
    return predictions
    
def f_score(y_true, y_pred):
    true_positives = sum(1 for true, pred in zip(y_true, y_pred) if true == pred == 1)
    false_positives = sum(1 for true, pred in zip(y_true, y_pred) if true == 0 and pred == 1)
    false_negatives = sum(1 for true, pred in zip(y_true, y_pred) if true == 1 and pred == 0)

    precision = true_positives / (true_positives + false_positives) if true_positives + false_positives > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if true_positives + false_negatives > 0 else 0
    
    if precision + recall == 0:
        return 0
    return 2 * (precision * recall) / (precision + recall)




if __name__ == "__main__":
    x_train, y_train = load_data("./SPAM_training_set/")
    model = nb_train(x_train, y_train)
    x_test, y_test = load_data("./SPAM_test_set/")
    configurations = [
        {"use_log": False, "smoothing": False},
        {"use_log": False, "smoothing": True},
        {"use_log": True, "smoothing": False},
        {"use_log": True, "smoothing": True}
    ]
    results = {}
    for config in configurations:
        y_pred = nb_test(x_test, model, use_log=config["use_log"], smoothing=config["smoothing"])
        f1 = f_score(y_test, y_pred)
        results[f"Log: {config['use_log']}, Smoothing: {config['smoothing']}"] = f1

    for config, f1_score in results.items():
        print(f"{config} -> F1-score: {f1_score:.4f}")

  #x_train, y_train = load_data("./SPAM_training_set/")
 # model = nb_train(x_train, y_train)
 # x_test, y_test = load_data("./SPAM_test_set/")
 # y_pred = nb_test(x_test, model, use_log = False, smoothing = False)
#  print(f_score(y_test,y_pred))




In [ ]:
import glob
import os
import numpy as np
from sklearn.metrics import f1_score

def load_data(directory):
    """
    Loads the email data from the specified directory.

    Args:
      directory: The directory containing the email files.

    Returns:
      A tuple containing two lists: x (documents) and y (labels).
    """
    x = []
    y = []
    for f in glob.glob(os.path.join(directory, "HAM.*.txt")):
        with open(f, 'r') as file:
            x.append(file.read())
            y.append(0)
    for f in glob.glob(os.path.join(directory, "SPAM.*.txt")):
        with open(f, 'r') as file:
            x.append(file.read())
            y.append(1)
    return x, y

def nb_train(x, y):
    """
    Trains a Naive Bayes classifier on the given data.

    Args:
      x: A list of documents (strings).
      y: A list of labels (0 for HAM, 1 for SPAM).

    Returns:
      A dictionary representing the trained model.
    """
    model = {
        'ham_count': 0,
        'spam_count': 0,
        'ham_fd': {},
        'spam_fd': {}
    }

    for label in y:
        if label == 0:
            model['ham_count'] += 1
        else:
            model['spam_count'] += 1

    for i, doc in enumerate(x):
        label = y[i]
        words = doc.split()
        for word in words:
            if label == 0:
                model['ham_fd'][word] = model['ham_fd'].get(word, 0) + 1
            else:
                model['spam_fd'][word] = model['spam_fd'].get(word, 0) + 1

    return model

def nb_test(docs, model, use_log=False, smoothing=False):
    predictions = []
    vocab_size = len(set(model['ham_fd']).union(model['spam_fd']))
    smoothing_val = 1 if smoothing else 0  # Use 1 for smoothing, 0 otherwise.

    for doc in docs:
        words = doc.split()
        # Initialize probabilities using the log of prior probabilities if using logs.
        if use_log:
            ham_prob = np.log((model['ham_count'] + smoothing_val) / (sum(model['ham_fd'].values()) + vocab_size * smoothing_val))
            spam_prob = np.log((model['spam_count'] + smoothing_val) / (sum(model['spam_fd'].values()) + vocab_size * smoothing_val))
        else:
            ham_prob = 1.0
            spam_prob = 1.0

        for word in words:
            ham_count = model['ham_fd'].get(word, 0) + smoothing_val
            spam_count = model['spam_fd'].get(word, 0) + smoothing_val
            if use_log:
                if ham_count > 0:  # Avoid log(0) by checking counts
                    ham_prob += np.log(ham_count / (model['ham_count'] + vocab_size * smoothing_val))
                if spam_count > 0:  # Avoid log(0)
                    spam_prob += np.log(spam_count / (model['spam_count'] + vocab_size * smoothing_val))
            else:
                ham_prob *= ham_count / (model['ham_count'] + vocab_size * smoothing_val)
                spam_prob *= spam_count / (model['spam_count'] + vocab_size * smoothing_val)

        prediction = 0 if ham_prob > spam_prob else 1
        predictions.append(prediction)

    return predictions


def calculate_f_score(y_true, y_pred):
    """
    Calculates the F1-score for the given predictions.

    Args:
      y_true: A list of true labels.
      y_pred: A list of predicted labels.

    Returns:
      The F1-score.
    """
    return f1_score(y_true, y_pred)

if __name__ == "__main__":
    # Load the training and test data
    x_train, y_train = load_data("./SPAM_training_set/")
    x_test, y_test = load_data("./SPAM_test_set/")

    # Train the model
    model = nb_train(x_train, y_train)

    # Run the experiments
    print("F1-score without log and without smoothing:", calculate_f_score(y_test, nb_test(x_test, model, use_log=False, smoothing=False)))
    print("F1-score with log and without smoothing:", calculate_f_score(y_test, nb_test(x_test, model, use_log=True, smoothing=False)))
    print("F1-score without log and with smoothing:", calculate_f_score(y_test, nb_test(x_test, model, use_log=False, smoothing=True)))
    print("F1-score with log and with smoothing:", calculate_f_score(y_test, nb_test(x_test, model, use_log=True, smoothing=True)))
